In [ ]:
import sys
sys.path.append('../../src/')

import matplotlib.pyplot as plt

# Import methods
import QGP.Defocusing.Methods as Def
import QGP.Focusing.Methods as Foc
from QGP.Soliton_Gen import *
from Utilities import *
import scipy.sparse as sp

import config

## Dark

In [ ]:
# Create solitons
L = 20
N = 5_001
k = 1/4

pars = {"k":k}

c_1 = 0
c_2 = np.sqrt(2)/2
c_3 = 0.1

# Theretical solitons
w_dark, x_dark = soliton(c_2, k, soliton_type="dark", L=L, N=N, as_func=True) #c=0

In [ ]:
#path = "test/"
Nt = 500
tf = 10

w_comp = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_comp.set_init_condition(w_dark)
w_comp.solve(method="fixed point")

In [ ]:
w_comp.animate_solution("test.gif", 50, auto_adjust=False, coord="real")

In [ ]:
w_mass = w_comp.get_mass(plot=True)
w_energy = w_comp.get_energy(plot=True)

In [ ]:
w_comp.plot(tf, label="Simulation")
plt.plot(x_dark, norm(w_dark(x_dark-c_2*tf)), label="Theoretical")
plt.legend()
plt.show()

## Bright

In [ ]:
L = 40
N = 5_000 #15_001
k = 1/4

pars = {"k":k}
c = np.sqrt(2)/2
w = 1 #np.sqrt(2)/2

w_bright, x_bright = soliton(c, k, w=w, soliton_type="bright", L=L, N=N+1, as_func=True)

In [ ]:
tf = 10
Nt = 160 

w_comp = Foc.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_comp.set_init_condition(w_bright)
w_comp.solve()#method="fixed point")

In [ ]:
w_comp.animate_solution("test.gif", 10)

In [ ]:
def bright_soliton(x, t, c=c, w=w):
    i = 1j
    output = w_bright(x-c*t) * np.exp(i*(c**2/4*t-c/2*x-w*t))
    return output

w_comp.w_theo = bright_soliton
error = w_comp.get_error()

In [ ]:
w_comp.plot(tf, label="Num")
plt.plot(x_bright, norm(bright_soliton(x_bright, tf)), label="Theo")
plt.legend()
plt.show()

In [ ]:
w_mass = w_comp.get_mass(plot=True)
w_energy = w_comp.get_energy(plot=True)

In [ ]:
plt.plot(w_comp.t, w_energy)
plt.plot(w_comp.t, w_mass)
#plt.ylim(1.3, 1.6)
plt.show()

## Gaussian init conditions

In [ ]:
A = 0.65
sigma = 1/10
def gaussian(x):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 10_000
tf = 2.25e-3
pars = {"k":k}

w_gauss = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss.set_init_condition(gaussian)
w_gauss.solve()#method="fixed point")

In [ ]:
N_2 = 5_000

w_gauss_N_2 = Def.CNFD(L, N_2, Nt, tf, pars, "neumann", order=2)
w_gauss_N_2.set_init_condition(gaussian)
w_gauss_N_2.solve()

In [ ]:
e_gauss = w_gauss.get_energy(background=0)
e_gauss_N_2 = w_gauss_N_2.get_energy(background=0)

In [ ]:
mass_gauss = w_gauss.get_mass()
mass_gauss_N_2 = w_gauss_N_2.get_mass()

In [ ]:
save_file(mass_gauss, "mass_N_10000")
save_file(mass_gauss_N_2, "mass_N_5000")

In [ ]:
w_gauss_max = np.abs(w_gauss.w).max(axis=1)
w_gauss_max_N_2 = np.abs(w_gauss_N_2.w).max(axis=1)

In [ ]:
plt.plot(w_gauss.t, w_gauss_max, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$N=10000$")
plt.plot(w_gauss_N_2.t, w_gauss_max_N_2, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$N=5000$")
plt.hlines(np.sqrt(2)/2, 0, w_gauss.t[-1], color="gray", ls="--")
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
plt.xlim(2e-3, 2.25e-3)
plt.legend()
#plt.ylim(0.65, 1)
#plt.savefig("gaussian.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# Other tests